# Taller 3 - EDA de FreeToGame
Lectura de datos crudos desde MongoDB, selección de variables y análisis exploratorio.

## 1. Conexión a MongoDB y lectura de la colección

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from pymongo import MongoClient

MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")
col = MongoClient(MONGO_URI)["taller3_db"]["raw_data"]
print("Documentos en taller3_db.raw_data:", col.count_documents({}))

## 2. Selección de variables
Se cargan los documentos y se conservan solo 7 columnas relevantes: id, título, género, plataforma, publisher, desarrollador y fecha de lanzamiento.

In [ ]:
docs = list(col.find({}, {"title": 1, "genre": 1, "platform": 1,
                         "publisher": 1, "developer": 1, "release_date": 1}))
df = pd.DataFrame(docs).rename(columns={"_id": "id"})
df = df[["id", "title", "genre", "platform", "publisher", "developer", "release_date"]]
df.shape

## 3. Limpieza
Se convierte `release_date` a fecha, se crea la columna `year` y se limpian espacios en texto.

In [ ]:
for c in ["title", "genre", "platform", "publisher", "developer"]:
    df[c] = df[c].astype(str).str.strip()
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["year"] = df["release_date"].dt.year
df.head()

## 4. Análisis exploratorio
### A. Inspección básica

In [ ]:
df.head()

In [ ]:
df.info()

Conteo de valores nulos por columna:

In [ ]:
df.isnull().sum()

### B. Cinco insights numéricos

In [ ]:
# Insight 1: total de juegos y género más frecuente
print(f"1) Hay {len(df)} juegos; el género más común es '{df['genre'].value_counts().idxmax()}' "
      f"con {df['genre'].value_counts().max()} juegos.")

# Insight 2: publisher con más juegos
pub = df["publisher"].value_counts()
print(f"2) El publisher con más juegos es '{pub.idxmax()}' con {pub.max()} títulos.")

# Insight 3: distribución por plataforma
plat = df["platform"].value_counts(normalize=True) * 100
print("3) Distribución por plataforma (%):")
print(plat.round(1).to_string())

# Insight 4: año con más lanzamientos
yr = df["year"].value_counts()
print(f"4) El año con más lanzamientos es {int(yr.idxmax())} con {yr.max()} juegos.")

# Insight 5: año promedio y rango de lanzamiento
print(f"5) Año promedio de lanzamiento: {df['year'].mean():.1f} "
      f"(más antiguo: {int(df['year'].min())}, más reciente: {int(df['year'].max())}).")

### C. Visualizaciones
**Gráfico 1 (torta):** proporción de juegos por género (los 6 principales + 'Otros').

In [ ]:
g = df["genre"].value_counts()
top = g.head(6)
if len(g) > 6:
    top["Otros"] = g.iloc[6:].sum()
plt.figure(figsize=(8, 8))
plt.pie(top, labels=top.index, autopct="%1.1f%%", startangle=90)
plt.title("Proporción de juegos por género")
plt.show()

**Gráfico 2 (barras):** los 10 publishers con más juegos.

In [ ]:
top_pub = df["publisher"].value_counts().head(10).sort_values()
plt.figure(figsize=(9, 5))
plt.barh(top_pub.index, top_pub.values, color="steelblue")
plt.title("Top 10 publishers por número de juegos")
plt.xlabel("Cantidad de juegos")
plt.ylabel("Publisher")
plt.tight_layout()
plt.show()

**Gráfico 3 (histograma):** distribución de juegos por año de lanzamiento.

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(df["year"].dropna(), bins=range(int(df["year"].min()), int(df["year"].max()) + 2),
         color="darkorange", edgecolor="black")
plt.title("Juegos lanzados por año")
plt.xlabel("Año de lanzamiento")
plt.ylabel("Cantidad de juegos")
plt.show()

## 5. Conclusiones
Escribe aquí (y en el PDF) un párrafo con lo que descubriste a partir de los 5 insights.